**Abstract**: This notebook outlines a production-grade Natural Language Processing (NLP) pipeline that fulfills the MASTER IMPLEMENTATION PLAN:

**dataset → preprocessing → intent classification → semantic retrieval → evidence grounding → evaluation → application**

### 1. Configuration & Reproducibility
Setting up global states and paths.

In [ ]:
import random
import numpy as np
import logging
from pathlib import Path

# Configure standard logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

class Config:
    DATA_DIR = Path("data")
    MODELS_DIR = Path("models")
    REPORTS_DIR = Path("reports")
    NGRAM_RANGE = (1, 2)
    MIN_DF = 3
    C_PARAM = 2.5

Config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
Config.REPORTS_DIR.mkdir(parents=True, exist_ok=True)
logger.info("Project initialized.")

### 2. Dataset Ingestion (Dataset)
Loading real-world university queries.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

dataset_path = Config.DATA_DIR / "real_world_university_queries.csv"
if os.path.exists(dataset_path):
    logger.info("Loading Real-World CLINC150 dataset...")
    df = pd.read_csv(dataset_path)
else:
    logger.info("Dataset not found, creating a mock dataset for demonstration...")
    df = pd.DataFrame({
        'query': ['where is the library', 'when are final exams', 'what is the passing grade', 'how do i register for courses', 'library hours'],
        'intent': ['location', 'schedule', 'policy', 'registration', 'location']
    })

df['query'] = df['query'].fillna("").astype(str)

# If using mock, stratify might fail with too few samples
try:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df['intent'])
except ValueError:
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)

logger.info(f"Ingested {len(train_df)} training and {len(test_df)} testing samples.")
display(train_df.head())

### 3. Exploratory Data Analysis & Text Preprocessing (Preprocessing)
Domain-aware normalization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.figure(figsize=(10, 4))
sns.countplot(data=train_df, y='intent', palette='viridis')
plt.title('Target Classes Distribution')
plt.tight_layout()
plt.show()

# Preprocessing
import re
def preprocess_query(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

X_train = [preprocess_query(q) for q in train_df['query'].tolist()]
X_test = [preprocess_query(q) for q in test_df['query'].tolist()]
logger.info(f"Raw Sample: {train_df['query'].iloc[0]} -> Normalized: {X_train[0]}")

### 4. Intent Classification (Intent Classification)
Label encoding, feature engineering, and training Logistic Regression.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['intent'].astype(str))
y_test = label_encoder.transform(test_df['intent'].astype(str))

# In case dataset is extremely small (mock), we lower min_df
min_df = Config.MIN_DF if len(train_df) > 10 else 1
vectorizer = TfidfVectorizer(ngram_range=Config.NGRAM_RANGE, min_df=min_df, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(C=Config.C_PARAM, max_iter=500, random_state=SEED, solver="lbfgs")
clf.fit(X_train_vec, y_train)

joblib.dump(vectorizer, Config.MODELS_DIR / "tfidf_vectorizer.joblib")
joblib.dump(clf, Config.MODELS_DIR / "intent_classifier.joblib")
joblib.dump(label_encoder, Config.MODELS_DIR / "label_encoder.joblib")

logger.info("Model artifacts serialized successfully.")

### 5. Semantic Retrieval (Semantic Retrieval)
Indexing university documents to find relevant context for grounded answers.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Creating a mock Knowledge Base representing university documents
knowledge_base = [
    "The university library is located in the central plaza and is open 24/7.",
    "Final exams are scheduled during the last two weeks of the academic semester.",
    "The passing grade for all undergraduate courses is a C (70%).",
    "Students can register for courses through the online portal starting August 1st."
]

# A separate vectorizer for semantic retrieval (could also use SentenceTransformers)
retrieval_vectorizer = TfidfVectorizer(stop_words='english')
kb_vectors = retrieval_vectorizer.fit_transform(knowledge_base)

def retrieve_evidence(query):
    q_vec = retrieval_vectorizer.transform([preprocess_query(query)])
    similarities = cosine_similarity(q_vec, kb_vectors).flatten()
    best_idx = np.argmax(similarities)
    
    # If no similarity, return a fallback
    if similarities[best_idx] == 0:
        return "No specific information found in the academic handbook.", 0.0
    return knowledge_base[best_idx], similarities[best_idx]

logger.info("Semantic retrieval engine initialized with knowledge base.")

### 6. Evidence Grounding & Generation (Evidence Grounding)
Fusing the predicted intent with the retrieved evidence to formulate an academically defensible answer.

In [ ]:
def generate_grounded_response(user_query):
    # Step 1: Preprocess
    q_clean = preprocess_query(user_query)
    
    # Step 2: Intent Classification
    q_vec = vectorizer.transform([q_clean])
    pred_idx = clf.predict(q_vec)[0]
    intent = label_encoder.inverse_transform([pred_idx])[0]
    
    # Step 3: Semantic Retrieval
    evidence, score = retrieve_evidence(user_query)
    
    # Step 4: Grounding (In a full RAG, this would go to an LLM. Here, we construct it deterministically)
    if score > 0.1:
        response = f"Based on the official documents (Intent: {intent}), {evidence}"
    else:
        response = f"I understand you are asking about {intent}, but I couldn't find specific evidence in the handbook."
        
    return {
        "query": user_query,
        "intent": intent,
        "evidence": evidence,
        "response": response,
        "confidence": score
    }

### 7. Evaluation (Evaluation)
Evaluating the underlying intent classification system.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

y_pred = clf.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)
print(f"OVERALL TEST ACCURACY: {accuracy:.4f}\n")

if len(label_encoder.classes_) > 1:
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, cmap="Blues", annot=True, fmt="d", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
    plt.title("Confusion Matrix")
    plt.show()

### 8. Application Demo (Application)
Simulating the final end-to-end system on unseen student queries.

In [ ]:
unseen_queries = [
    "Where is the main library?",
    "What date do final exams start?",
    "What is the minimum grade to pass?",
    "How do I add a new course?",
    "Is there a gym on campus?"
]

for q in unseen_queries:
    print(f"\n{'='*50}")
    print(f"STUDENT QUERY: {q}")
    result = generate_grounded_response(q)
    print(f"PREDICTED INTENT: {result['intent']}")
    print(f"RETRIEVED EVIDENCE: {result['evidence']} (Score: {result['confidence']:.2f})")
    print(f"FINAL GROUNDED RESPONSE: {result['response']}")
print(f"\n{'='*50}")
logger.info("Application demonstration complete.")